# Anomaly Detection, Quality Audit & Dataset Sanitization
### Automated Identification of 8 Failure Modes + Exporting Clean vs. Anomaly Splits

This notebook audits training and evaluation annotations to detect faulty labeling and anomalies:
1. **Degenerate Boxes** ($w \le 0$ or $h \le 0$)
2. **Micro-Boxes / Stray Clicks** ($w < 12$ or $h < 12$ or area ratio $< 0.0001$)
3. **Giant / Full-Image Boxes** ($	ext{area} > 65\%$)
4. **Extreme Aspect Ratio Slivers** ($	ext{aspect} > 7.0$ or $< 0.15$)
5. **Out-of-Bounds Coordinates** ($x < -5$, $y < -5$, $x+w > 	ext{img\_w}+10$, $y+h > 	ext{img\_h}+10$)
6. **Duplicate / Highly Overlapping Annotations** ($	ext{IoU} > 0.70$)
7. **Loose / Oversized Box Around Small Tag** (excess shelf padding or disproportionate height)
8. **Dense Shelf Clusters** ($> 12$ tags)

---

### Crucial Deliverables:
- **`anomalies_manifest.json`** & **`anomalies_report.csv`**: Full audit log of flagged issues.
- **`cleaned_dataset/`**: Sanitized COCO annotations **with all anomalies excluded** for model training and clean evaluation.
- **`anomaly_dataset/`**: Isolated anomaly split for robustness testing.

In [ ]:
# STEP 0: Dependencies
import sys

%pip install -q pycocotools Pillow matplotlib pandas numpy tabulate tqdm

print("✓ Anomaly detector dependencies ready.")

In [ ]:
# CELL 1: Imports & Helper Functions
import os
import sys
import json
import logging
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("AnomalyAuditor")

def compute_iou(box1, box2):
    # box format: [x, y, w, h]
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xa = max(x1, x2)
    ya = max(y1, y2)
    xb = min(x1 + w1, x2 + w2)
    yb = min(y1 + h1, y2 + h2)
    inter = max(0.0, xb - xa) * max(0.0, yb - ya)
    area1 = max(0.0, w1) * max(0.0, h1)
    area2 = max(0.0, w2) * max(0.0, h2)
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

print("Helpers and imports configured.")

In [ ]:
# CELL 2: Discovery of Input Annotations
# Resilient workspace root resolver
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists()), Path("..").resolve())
CANDIDATE_PATHS = [
    REPO_ROOT / "coco_files" / "_annotations.coco.json",
    REPO_ROOT / "coco_files",
    REPO_ROOT / "multi_class_train_rfdetr" / "dataset_full_data" / "train" / "_annotations.coco.json",
    REPO_ROOT / "multi_class_train_rfdetr" / "dataset_sample_1000" / "train" / "_annotations.coco.json",
    REPO_ROOT / "dataset_full_data" / "train" / "_annotations.coco.json"
]

SRC_ANN_FILE = None
for p in CANDIDATE_PATHS:
    if p.is_file() and p.stat().st_size > 0:
        SRC_ANN_FILE = p
        break
    elif p.is_dir():
        js = list(p.glob("*.json"))
        if js:
            SRC_ANN_FILE = js[0]
            break

if SRC_ANN_FILE is None:
    SRC_ANN_FILE = REPO_ROOT / "coco_files" / "_annotations.coco.json"

print(f"Target Source Annotations: {SRC_ANN_FILE}")

OUTPUT_CLEAN_DIR = Path("cleaned_dataset")
OUTPUT_ANOM_DIR = Path("anomaly_dataset")
OUTPUT_CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ANOM_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# CELL 3: Load COCO Data
raw_coco = {"images": [], "annotations": [], "categories": []}
if SRC_ANN_FILE.exists():
    with open(SRC_ANN_FILE, "r") as f:
        raw_coco = json.load(f)
    print(f"Loaded {len(raw_coco['images'])} images, {len(raw_coco['annotations'])} annotations.")
else:
    print(f"Notice: Source file '{SRC_ANN_FILE}' not found yet. Anomaly detector ready for when file is placed.")

cat_id_to_name = {c["id"]: c["name"] for c in raw_coco.get("categories", [])}
anns_by_img = defaultdict(list)
for a in raw_coco.get("annotations", []):
    anns_by_img[a["image_id"]].append(a)

In [ ]:
# CELL 4: Execute 8 Anomaly Detection Audits
flagged_anomalies = []
flagged_ann_ids = set()
flagged_img_ids = set()

for img in raw_coco.get("images", []):
    img_id = img["id"]
    fname = img.get("file_name", f"img_{img_id}.jpg")
    img_w = float(img.get("width", 1008))
    img_h = float(img.get("height", 1008))
    img_area = img_w * img_h
    anns = anns_by_img.get(img_id, [])
    
    # 1. Overcrowded (>12 tags)
    if len(anns) > 12:
        flagged_anomalies.append({
            "image_id": img_id, "file_name": fname, "ann_id": None,
            "category": "multiple", "issue_type": "Overcrowded Cluster (>12 tags)", "severity": "MEDIUM"
        })
        flagged_img_ids.add(img_id)
        
    valid_boxes = [a["bbox"] for a in anns if len(a.get("bbox", [])) == 4 and a["bbox"][2] > 0 and a["bbox"][3] > 0]
    med_h = np.median([b[3] for b in valid_boxes]) if valid_boxes else 35.0
    
    for i, ann in enumerate(anns):
        ann_id = ann["id"]
        cname = cat_id_to_name.get(ann["category_id"], "tag")
        b = ann.get("bbox", [0, 0, 0, 0])
        x, y, w, h = float(b[0]), float(b[1]), float(b[2]), float(b[3])
        area = w * h
        aspect = w / max(1.0, h)
        
        # 2. Degenerate box
        if w <= 0 or h <= 0:
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id,
                "category": cname, "issue_type": "Degenerate Box (w<=0 or h<=0)", "severity": "CRITICAL"
            })
            flagged_ann_ids.add(ann_id)
            flagged_img_ids.add(img_id)
            continue
            
        # 3. Micro-Box / Stray Click
        if w < 12 or h < 12 or (area / img_area < 0.0001):
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id,
                "category": cname, "issue_type": "Micro-Box / Stray Click (<12px)", "severity": "HIGH"
            })
            flagged_ann_ids.add(ann_id)
            flagged_img_ids.add(img_id)
            
        # 4. Giant / Full-Image Box
        if area / img_area > 0.65:
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id,
                "category": cname, "issue_type": "Giant Full-Image Box (>65%)", "severity": "HIGH"
            })
            flagged_ann_ids.add(ann_id)
            flagged_img_ids.add(img_id)
            
        # 5. Extreme Aspect Ratio
        if aspect > 7.0 or aspect < 0.15:
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id,
                "category": cname, "issue_type": "Extreme Aspect Ratio Sliver", "severity": "MEDIUM"
            })
            flagged_ann_ids.add(ann_id)
            flagged_img_ids.add(img_id)
            
        # 6. Out of Bounds
        if x < -5 or y < -5 or (x + w) > (img_w + 10) or (y + h) > (img_h + 10):
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id,
                "category": cname, "issue_type": "Out of Bounds Box", "severity": "HIGH"
            })
            flagged_ann_ids.add(ann_id)
            flagged_img_ids.add(img_id)
            
        # 7. Duplicate / Overlap
        for j in range(i + 1, len(anns)):
            ann2 = anns[j]
            b2 = ann2.get("bbox", [0, 0, 0, 0])
            if compute_iou(b, b2) > 0.70:
                flagged_anomalies.append({
                    "image_id": img_id, "file_name": fname, "ann_id": ann_id,
                    "category": cname, "issue_type": "Duplicate / Overlapping Box (IoU>0.70)", "severity": "CRITICAL"
                })
                flagged_ann_ids.add(ann_id)
                flagged_img_ids.add(img_id)
                
        # 8. Loose / Oversized Box
        if aspect < 1.35 and h > 22:
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id,
                "category": cname, "issue_type": "Loose Box (Excess Shelf Padding)", "severity": "HIGH"
            })
            flagged_ann_ids.add(ann_id)
            flagged_img_ids.add(img_id)

df_anomalies = pd.DataFrame(flagged_anomalies)
print(f"✓ Audit completed: {len(flagged_anomalies)} anomalies flagged across {len(flagged_img_ids)} images.")
if not df_anomalies.empty:
    print(df_anomalies["issue_type"].value_counts())

In [ ]:
# CELL 5: Export Anomalies Manifest & Audit Report
manifest_path = Path("anomalies_manifest.json")
report_path = Path("anomalies_report.csv")

manifest_data = {
    "total_anomalies": len(flagged_anomalies),
    "flagged_image_ids": list(flagged_img_ids),
    "flagged_annotation_ids": list(flagged_ann_ids),
    "summary_by_type": df_anomalies["issue_type"].value_counts().to_dict() if not df_anomalies.empty else {}
}

with open(manifest_path, "w") as f:
    json.dump(manifest_data, f, indent=2)
if not df_anomalies.empty:
    df_anomalies.to_csv(report_path, index=False)

print(f"✓ Saved anomalies manifest: {manifest_path.resolve()}")
print(f"✓ Saved anomalies CSV report: {report_path.resolve()}")

In [ ]:
# CELL 6: Export Clean Dataset (Excluding Anomalies for Training & Clean Testing)
# Strips all flagged anomaly annotations and invalid images

clean_images = [img for img in raw_coco.get("images", []) if img["id"] not in flagged_img_ids]
clean_annotations = [ann for ann in raw_coco.get("annotations", []) if ann["id"] not in flagged_ann_ids and ann["image_id"] not in flagged_img_ids]

clean_coco = {
    "info": raw_coco.get("info", {"description": "Sanitized Clean Dataset (Anomalies Excluded)", "version": "1.0"}),
    "licenses": raw_coco.get("licenses", []),
    "images": clean_images,
    "annotations": clean_annotations,
    "categories": raw_coco.get("categories", [])
}

clean_ann_path = OUTPUT_CLEAN_DIR / "_annotations.coco.json"
with open(clean_ann_path, "w") as f:
    json.dump(clean_coco, f, indent=2)

# Also export anomaly-only split for downstream robustness auditing
anom_images = [img for img in raw_coco.get("images", []) if img["id"] in flagged_img_ids]
anom_annotations = [ann for ann in raw_coco.get("annotations", []) if ann["id"] in flagged_ann_ids or ann["image_id"] in flagged_img_ids]

anom_coco = {
    "info": {"description": "Anomaly Evaluation Split", "version": "1.0"},
    "licenses": raw_coco.get("licenses", []),
    "images": anom_images,
    "annotations": anom_annotations,
    "categories": raw_coco.get("categories", [])
}

anom_ann_path = OUTPUT_ANOM_DIR / "_annotations.coco.json"
with open(anom_ann_path, "w") as f:
    json.dump(anom_coco, f, indent=2)

print("==============================================================================")
print(" DATASET SANITIZATION SUMMARY")
print("==============================================================================")
print(f" Total Raw Images:          {len(raw_coco.get('images', []))}")
print(f" Clean Images (Retained):   {len(clean_images)}")
print(f" Flagged Anomaly Images:    {len(anom_images)}")
print(f" Total Raw Annotations:     {len(raw_coco.get('annotations', []))}")
print(f" Clean Annotations:         {len(clean_annotations)}")
print(f" Flagged Anomaly Boxes:     {len(anom_annotations)}")
print(f" Clean Dataset Saved To:    {clean_ann_path.resolve()}")
print(f" Anomaly Dataset Saved To:  {anom_ann_path.resolve()}")
print("==============================================================================")